In [123]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv('railway.csv')
df.columns = [col.strip().replace(' ', '_') for col in df.columns]


df['Railcard'] = df['Railcard'].fillna('None')
df['Reason_for_Delay'] = df['Reason_for_Delay'].fillna('No Delay')

delay_map = {
    'Signal failure': 'Signal Failure',
    'Weather Conditions': 'Weather',
    'Staff Shortage': 'Staffing',
}
df['Reason_for_Delay'] = df['Reason_for_Delay'].replace(delay_map)

mask = (
    (df['Arrival_Time'] == df['Actual_Arrival_Time']) &
    (df['Journey_Status'] == 'Delayed')
)
df.loc[mask, 'Journey_Status'] = 'On Time'
df.loc[mask, 'Reason_for_Delay'] = 'No Delay'

df['Date_of_Purchase'] = pd.to_datetime(df['Date_of_Purchase'])
df['Date_of_Journey']  = pd.to_datetime(df['Date_of_Journey'])

dept = pd.to_datetime(df['Date_of_Journey'].astype(str) + ' ' + df['Departure_Time'])
sched_arr = pd.to_datetime(df['Date_of_Journey'].astype(str) + ' ' + df['Arrival_Time'])
act_arr = pd.to_datetime(df['Date_of_Journey'].astype(str) + ' ' + df['Actual_Arrival_Time'], errors='coerce')

sched_arr[sched_arr < dept] += pd.Timedelta(days=1)
act_arr[(act_arr.notnull()) & (act_arr < dept)] += pd.Timedelta(days=1)

df['Journey_Duration_Minutes'] = (sched_arr - dept).dt.total_seconds() / 60
df['Delay_Minutes'] = (act_arr - sched_arr).dt.total_seconds() / 60

df.loc[df['Journey_Status'] == 'On Time', 'Delay_Minutes'] = 0
df.loc[df['Journey_Status'] == 'Cancelled', 'Delay_Minutes'] = np.nan

df['Peak_Category'] = np.where(
    dept.dt.hour.isin([7,8,9,16,17,18]),
    'Peak','Off-Peak'
)


def calc_refund(row):
    if row['Journey_Status'] == 'Cancelled' and row['Refund_Request'] == 'Yes':
        return row['Price']
    if row['Ticket_Type'] == 'Advance':
        return 0
    if row['Journey_Status'] == 'Delayed' and row['Refund_Request'] == 'Yes':
        mins = row['Delay_Minutes']
        if mins >= 60: return row['Price']
        elif mins >= 30: return row['Price'] * 0.50
        elif mins >= 15: return row['Price'] * 0.25
    return 0

df['Refunded_Amount'] = df.apply(calc_refund, axis=1)
df['Profit'] = df['Price'] - df['Refunded_Amount']

routes = df[['Departure_Station','Arrival_Destination']].drop_duplicates().reset_index(drop=True)
routes = routes.rename(columns={'Arrival_Destination':'Arrival_Station'})
routes['Route_ID'] = routes.index + 1


df = df.rename(columns={'Arrival_Destination': 'Arrival_Station'})

df = df.merge(routes, on=['Departure_Station','Arrival_Station'], how='left')


trips = df.groupby(['Route_ID','Date_of_Journey','Departure_Time']).agg({
    'Arrival_Time':'first',
    'Actual_Arrival_Time':'first',
    'Journey_Duration_Minutes':'mean',
    'Delay_Minutes':'mean',
    'Peak_Category':'first',
    'Reason_for_Delay':lambda x: x.mode()[0] if not x.mode().empty else 'No Delay',
    'Journey_Status':lambda x: 'Delayed' if 'Delayed' in x.values
                             else ('Cancelled' if 'Cancelled' in x.values
                             else 'On Time')
}).reset_index()

trips['Trip_ID'] = range(1, len(trips)+1)

df = df.merge(
    trips[['Route_ID','Date_of_Journey','Departure_Time','Trip_ID']],
    on=['Route_ID','Date_of_Journey','Departure_Time'],
    how='left'
)


rc = df.copy()

rc['Journey_Date'] = pd.to_datetime(rc['Date_of_Journey'])
rc['Purchase_Date_dt'] = pd.to_datetime(rc['Date_of_Purchase'])


railway_cleaned = df.copy()
railway_cleaned = railway_cleaned[[
    'Transaction_ID',
    'Date_of_Purchase',
    'Time_of_Purchase',
    'Purchase_Type',
    'Payment_Method',
    'Railcard',
    'Ticket_Class',
    'Ticket_Type',
    'Price',
    'Departure_Station',
    'Arrival_Station',
    'Date_of_Journey',
    'Departure_Time',
    'Arrival_Time',
    'Actual_Arrival_Time',
    'Journey_Status',
    'Reason_for_Delay',
    'Delay_Minutes',
    'Journey_Duration_Minutes',
    'Peak_Category',
    'Refund_Request',
    'Refunded_Amount',
    'Profit'
]]
output_file = 'railway_analysis.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

    trips.to_excel(writer, sheet_name='Trips', index=False)
    df.to_excel(writer, sheet_name='Transactions', index=False)
    routes.to_excel(writer, sheet_name='Routes', index=False)

    railway_cleaned.to_excel(writer, sheet_name='Railway_Cleaned', index=False)

In [13]:
# Q1
print("\nQ1. Total journeys recorded:", trips['Trip_ID'].nunique())


Q1. Total journeys recorded: 19871


In [63]:
# Q2
trips['Year'] = trips['Date_of_Journey'].dt.year
trips['Month'] = trips['Date_of_Journey'].dt.month
trips['Day'] = trips['Date_of_Journey'].dt.day_name()

print("\n Yearly Journeys:")
print(trips.groupby('Year')['Trip_ID'].count())

print("\n Monthly Journeys:")
print(trips.groupby(['Year','Month'])['Trip_ID'].count())

print("\n Daily Journeys:")
print(trips.groupby('Date_of_Journey')['Trip_ID'].count().head(10))


 Yearly Journeys:
Year
2024    19871
Name: Trip_ID, dtype: int64

 Monthly Journeys:
Year  Month
2024  1        5060
      2        4839
      3        5083
      4        4889
Name: Trip_ID, dtype: int64

 Daily Journeys:
Date_of_Journey
2024-01-01     61
2024-01-02    112
2024-01-03    168
2024-01-04    176
2024-01-05    159
2024-01-06    165
2024-01-07    167
2024-01-08    174
2024-01-09    169
2024-01-10    164
Name: Trip_ID, dtype: int64


In [17]:
# Q3
print("\nQ3. Journey Status Distribution:")
print(trips.groupby('Journey_Status')['Trip_ID'].count())


Q3. Journey Status Distribution:
Journey_Status
Cancelled      790
Delayed       1048
On Time      18033
Name: Trip_ID, dtype: int64


In [18]:
# Q4
print("\nQ4. Avg Journey Duration:", trips['Journey_Duration_Minutes'].mean())


Q4. Avg Journey Duration: 70.20154999748377


In [19]:

# Q5
print("Q5. Avg Delay:", trips['Delay_Minutes'].mean())


Q5. Avg Delay: 2.0997851265656937


In [20]:
# Q6-8
print("\nQ6. Total Revenue:", rc['Price'].sum())
print("Q7. Total Refunds:", rc['Refunded_Amount'].sum())
print("Q8. Net Profit:", rc['Profit'].sum())


Q6. Total Revenue: 741921
Q7. Total Refunds: 15297.75
Q8. Net Profit: 726623.25


In [21]:
# Q9
print("\nQ9. Revenue by Ticket Type:")
print(rc.groupby('Ticket_Type')['Profit'].sum())


Q9. Revenue by Ticket Type:
Ticket_Type
Advance     303747.00
Anytime     205459.25
Off-Peak    217417.00
Name: Profit, dtype: float64


In [22]:
# Q10
print("\nQ10. Revenue by Ticket Class:")
print(rc.groupby('Ticket_Class')['Profit'].sum())


Q10. Revenue by Ticket Class:
Ticket_Class
First Class    146887.00
Standard       579736.25
Name: Profit, dtype: float64


In [23]:
# Q11
print("\nQ11. Revenue by Month:")
rc['Month'] = rc['Purchase_Date_dt'].dt.month
print(rc.groupby('Month')['Profit'].sum())


Q11. Revenue by Month:
Month
1     200343.25
2     151332.50
3     190573.75
4     183681.75
12       692.00
Name: Profit, dtype: float64


In [24]:
# Q12
print("\nQ12. Revenue by Route:")
print(rc.groupby('Route_ID')['Profit'].sum().head(10))


Q12. Revenue by Route:
Route_ID
1       2454.00
2     180219.00
3      11703.50
4      63841.00
5     110588.25
6         41.00
7        651.00
8       1935.00
9      17021.00
10     18762.00
Name: Profit, dtype: float64


In [25]:

# Q13
print("\nQ13. Avg Ticket Price per Route:")
print(rc.groupby('Route_ID')['Price'].mean().head(10))


Q13. Avg Ticket Price per Route:
Route_ID
1      55.772727
2      46.709077
3       3.980680
4      16.877872
5     103.280766
6       2.562500
7      38.294118
8       7.686047
9       3.740277
10     27.081197
Name: Price, dtype: float64


In [26]:
# Q14
print("\nQ14. Railcard Usage:")
print(rc.groupby('Railcard')['Price'].count())


Q14. Railcard Usage:
Railcard
Adult        4846
Disabled     3089
None        20918
Senior       2800
Name: Price, dtype: int64


In [27]:
# Q15
print("\nQ15. Payment Method Frequency:")
print(rc.groupby('Payment_Method')['Transaction_ID'].count())


Q15. Payment Method Frequency:
Payment_Method
Contactless    10834
Credit Card    19136
Debit Card      1683
Name: Transaction_ID, dtype: int64


In [28]:
# Q16
print("\nQ16. Revenue by Payment Method:")
print(rc.groupby('Payment_Method')['Profit'].sum())


Q16. Revenue by Payment Method:
Payment_Method
Contactless    215237.00
Credit Card    461529.50
Debit Card      49856.75
Name: Profit, dtype: float64


In [29]:
# Q17
print("\nQ17. Revenue by Purchase Type:")
print(rc.groupby('Purchase_Type')['Profit'].sum())


Q17. Revenue by Purchase Type:
Purchase_Type
Online     375390.50
Station    351232.75
Name: Profit, dtype: float64


In [30]:
# Q19
print("\nQ19. Ticket Demand Trend:")
print(rc.groupby(['Month','Ticket_Type'])['Transaction_ID'].count())


Q19. Ticket Demand Trend:
Month  Ticket_Type
1      Advance        4293
       Anytime        1640
       Off-Peak       2501
2      Advance        5328
       Anytime         755
       Off-Peak       1254
3      Advance        4002
       Anytime        1475
       Off-Peak       2623
4      Advance        3904
       Anytime        1470
       Off-Peak       2374
12     Advance          34
Name: Transaction_ID, dtype: int64


In [31]:
# Q20
print("\nQ20. Railcard Ticket Preferences:")
print(rc[rc['Railcard']!='None'].groupby('Ticket_Type')['Transaction_ID'].count())


Q20. Railcard Ticket Preferences:
Ticket_Type
Advance     5936
Anytime     1969
Off-Peak    2830
Name: Transaction_ID, dtype: int64


In [32]:

# Q21
print("\nQ21. Refund Request Rate:",
      (rc['Refund_Request']=='Yes').mean()*100)


Q21. Refund Request Rate: 3.5320506745016274


In [33]:
# Q22
print("\nQ22. Refunds by Ticket Type:")
print(rc[rc['Refund_Request']=='Yes'].groupby('Ticket_Type')['Transaction_ID'].count())


Q22. Refunds by Ticket Type:
Ticket_Type
Advance     611
Anytime     138
Off-Peak    369
Name: Transaction_ID, dtype: int64


In [34]:
# Q23
print("\nQ23. Delay Minutes by Route:")
print(trips.groupby('Route_ID')['Delay_Minutes'].sum().head(10))


Q23. Delay Minutes by Route:
Route_ID
1         0.0
2       682.0
3      1372.0
4      1415.0
5      5349.0
6         0.0
7       559.0
8       436.0
9     10644.0
10        0.0
Name: Delay_Minutes, dtype: float64


In [35]:
# Q24
print("\nQ24. Longest Routes:")
print(trips.groupby('Route_ID')['Journey_Duration_Minutes'].mean().sort_values(ascending=False).head(10))


Q24. Longest Routes:
Route_ID
48    270.0
46    260.0
43    260.0
1     150.0
32    150.0
45    150.0
57    150.0
58    150.0
55    150.0
35    135.0
Name: Journey_Duration_Minutes, dtype: float64


In [36]:
# Q25
print("\nQ25. Delays by Peak:")
print(trips.groupby('Peak_Category')['Delay_Minutes'].count())


Q25. Delays by Peak:
Peak_Category
Off-Peak    11910
Peak         7171
Name: Delay_Minutes, dtype: int64


In [37]:
# Q26
print("\nQ26 Delay likelihood:")
print(trips.groupby('Peak_Category')['Journey_Status'].apply(lambda x:(x=='Delayed').mean()*100))


Q26 Delay likelihood:
Peak_Category
Off-Peak    4.140050
Peak        7.142857
Name: Journey_Status, dtype: float64


In [38]:
# Q27
print("\nQ27 Delay by day:")
print(trips.groupby(trips['Date_of_Journey'].dt.day_name())['Journey_Status']
      .apply(lambda x:(x=='Delayed').mean()*100))


Q27 Delay by day:
Date_of_Journey
Friday       5.240015
Monday       5.670471
Saturday     5.030325
Sunday       5.047319
Thursday     5.290456
Tuesday      5.517241
Wednesday    5.121107
Name: Journey_Status, dtype: float64


In [39]:
# Q28
print("\nQ28 Longest:", trips['Journey_Duration_Minutes'].max())
print("Shortest:", trips['Journey_Duration_Minutes'].min())


Q28 Longest: 270.0
Shortest: 15.0


In [40]:
# Q29
print("\nQ29 Delay vs duration:")
print(trips.groupby(pd.cut(trips['Journey_Duration_Minutes'], bins=[0,30,60,999]))['Delay_Minutes'].mean())


Q29 Delay vs duration:
Journey_Duration_Minutes
(0, 30]      2.334279
(30, 60]     2.806194
(60, 999]    1.743640
Name: Delay_Minutes, dtype: float64


In [41]:
# Q30
print("\nQ30 On-time rate per route:")
print(trips.groupby('Route_ID')['Journey_Status'].apply(lambda x:(x=='On Time').mean()*100).head(10))


Q30 On-time rate per route:
Route_ID
1     100.000000
2      94.583333
3      93.692104
4      93.242123
5      39.393939
6     100.000000
7       0.000000
8      86.549708
9      90.093458
10     95.567867
Name: Journey_Status, dtype: float64


In [42]:
# Q31
print("\nQ31 Delayed vs Not:")
print(trips['Journey_Status'].value_counts())


Q31 Delayed vs Not:
Journey_Status
On Time      18033
Delayed       1048
Cancelled      790
Name: count, dtype: int64


In [43]:
# Q32
print("\nQ32 Delay reasons:")
print(trips['Reason_for_Delay'].value_counts())


Q32 Delay reasons:
Reason_for_Delay
No Delay           18033
Weather              462
Staffing             434
Signal Failure       424
Technical Issue      367
Traffic              151
Name: count, dtype: int64


In [44]:
# Q33
print("\nQ33 Delay time by reason:")
print(trips.groupby('Reason_for_Delay')['Delay_Minutes'].sum())


Q33 Delay time by reason:
Reason_for_Delay
No Delay               0.0
Signal Failure     13256.0
Staffing           10325.0
Technical Issue     5610.0
Traffic             1063.0
Weather             9812.0
Name: Delay_Minutes, dtype: float64


In [45]:
# Q34
print("\nQ34 Avg delay per reason:")
print(trips.groupby('Reason_for_Delay')['Delay_Minutes'].mean())


Q34 Avg delay per reason:
Reason_for_Delay
No Delay            0.000000
Signal Failure     61.087558
Staffing           42.665289
Technical Issue    21.412214
Traffic            20.843137
Weather            35.550725
Name: Delay_Minutes, dtype: float64


In [46]:
# Q35
print("\nQ35 Top delay reason per route:")
print(trips.groupby(['Route_ID','Reason_for_Delay'])['Trip_ID'].count().head(10))


Q35 Top delay reason per route:
Route_ID  Reason_for_Delay
1         No Delay              43
2         No Delay            2270
          Signal Failure        54
          Staffing              19
          Technical Issue       21
          Traffic               16
          Weather               20
3         No Delay            2124
          Signal Failure        32
          Staffing              12
Name: Trip_ID, dtype: int64


In [47]:

# Q36
print("\nQ36 Delay by hour:")
print(trips.groupby(pd.to_datetime(trips['Departure_Time']).dt.hour)['Delay_Minutes'].mean())


Q36 Delay by hour:
Departure_Time
0      0.955556
1      0.764822
2      0.000000
3      4.949115
4      0.384831
5      1.476974
6      0.822208
7      0.140458
8      6.169929
9     12.159502
10     0.000000
11    11.185455
12     0.000000
13     0.000000
14     0.435241
15     2.407407
16     1.524245
17     2.421610
18     0.000000
19     0.000000
20     0.000000
21     0.000000
22     0.000000
23     0.000000
Name: Delay_Minutes, dtype: float64


In [48]:
# Q37
print("\nQ37 Delay distribution:")
print(pd.cut(trips['Delay_Minutes'], bins=[0,30,60,999]).value_counts())


Q37 Delay distribution:
Delay_Minutes
(0, 30]      584
(30, 60]     332
(60, 999]    132
Name: count, dtype: int64


In [49]:

# Q38
print("\nQ38 Refund cost by reason:")
print(df.groupby('Reason_for_Delay')['Refunded_Amount'].sum())


Q38 Refund cost by reason:
Reason_for_Delay
No Delay              0.00
Signal Failure     3209.75
Staffing           4260.50
Technical Issue    3100.50
Traffic            1952.00
Weather            2775.00
Name: Refunded_Amount, dtype: float64


In [50]:
# Q39
print("\nQ39 % refunded:", (df['Refunded_Amount']>0).mean()*100)


Q39 % refunded: 2.322054781537295


In [51]:
# Q40
print("\nQ40 Total refund:", df['Refunded_Amount'].sum())


Q40 Total refund: 15297.75


In [53]:
# Q41
print("\nQ41 Refund rate by ticket:")
print(df.groupby('Ticket_Type')['Refunded_Amount'].apply(lambda x:(x>0).mean()*100))


Q41 Refund rate by ticket:
Ticket_Type
Advance     1.867775
Anytime     2.172285
Off-Peak    3.324954
Name: Refunded_Amount, dtype: float64


In [54]:
# Q42
print("\nQ42 Refund by delay range:")
print(df.groupby(pd.cut(df['Delay_Minutes'], bins=[0,30,60,999]))['Refunded_Amount'].count())


Q42 Refund by delay range:
Delay_Minutes
(0, 30]       937
(30, 60]     1014
(60, 999]     323
Name: Refunded_Amount, dtype: int64


In [55]:
# Q43
print("\nQ43 Route refund rate:")
print(df.groupby('Route_ID')['Refunded_Amount'].apply(lambda x:(x>0).mean()*100).head(10))


Q43 Route refund rate:
Route_ID
1     0.000000
2     1.784804
3     2.465023
4     2.297960
5     5.560620
6     0.000000
7     0.000000
8     2.325581
9     1.771824
10    1.709402
Name: Refunded_Amount, dtype: float64


In [56]:
# Q44
print("\nQ44 Avg refund:", df[df['Refunded_Amount']>0]['Refunded_Amount'].mean())


Q44 Avg refund: 20.81326530612245


In [57]:
# Q45
print("\nQ45 Refund impact:")
print(df.groupby('Journey_Status')['Refunded_Amount'].sum())


Q45 Refund impact:
Journey_Status
Cancelled    12537.00
Delayed       2760.75
On Time          0.00
Name: Refunded_Amount, dtype: float64


In [58]:
# Q46
print("\nQ46 Route efficiency:")
print(df.groupby('Route_ID').agg({'Profit':'sum','Delay_Minutes':'mean'}).head(10))


Q46 Route efficiency:
             Profit  Delay_Minutes
Route_ID                          
1           2454.00       0.000000
2         180219.00       0.566058
3          11703.50       0.811675
4          63841.00       0.655422
5         110588.25      28.588176
6             41.00       0.000000
7            651.00      36.352941
8           1935.00       2.020747
9          17021.00       5.484094
10         18762.00       0.000000


In [59]:
# Q47
print("\nQ47 On-time peak vs off:")
print(trips.groupby('Peak_Category')['Journey_Status'].apply(lambda x:(x=='On Time').mean()*100))


Q47 On-time peak vs off:
Peak_Category
Off-Peak    92.164632
Peak        88.419510
Name: Journey_Status, dtype: float64


In [60]:
# Q48
print("\nQ48 Peak performance:")
print(trips.groupby('Peak_Category')['Delay_Minutes'].mean())


Q48 Peak performance:
Peak_Category
Off-Peak    1.436020
Peak        3.202203
Name: Delay_Minutes, dtype: float64


In [61]:
# Q49
print("\nQ49 Revenue loss by reason:")
print(df.groupby('Reason_for_Delay')['Refunded_Amount'].sum())


Q49 Revenue loss by reason:
Reason_for_Delay
No Delay              0.00
Signal Failure     3209.75
Staffing           4260.50
Technical Issue    3100.50
Traffic            1952.00
Weather            2775.00
Name: Refunded_Amount, dtype: float64


In [62]:
# Q50
print("\nQ50 Best combo:")
print(df.groupby(['Route_ID','Peak_Category','Ticket_Type'])['Profit']
      .sum().sort_values(ascending=False).head(5))


Q50 Best combo:
Route_ID  Peak_Category  Ticket_Type
5         Peak           Anytime        48606.0
2         Peak           Anytime        40331.0
          Off-Peak       Advance        39487.0
5         Peak           Advance        39310.0
2         Off-Peak       Off-Peak       37104.0
Name: Profit, dtype: float64


In [65]:
#===============================================================

In [66]:
# Q1. How many unique routes are there?
print(routes['Route_ID'].nunique())

65


In [68]:
# Q2. Which routes have the highest number of trips?
routes['Route_Name'] = routes['Departure_Station'] + '_' + routes['Arrival_Station']
trips.merge(routes, on='Route_ID').groupby('Route_Name')['Trip_ID'].count().sort_values(ascending=False)

,Trip_ID
Route_Name,
Manchester Piccadilly_Liverpool Lime Street,2675
London Euston_Birmingham New Street,2515
London Paddington_Reading,2412
London Kings Cross_York,2400
Liverpool Lime Street_Manchester Piccadilly,2267
...,...
Liverpool Lime Street_Birmingham New Street,14
York_Edinburgh Waverley,14
Manchester Piccadilly_Warrington,13


In [69]:
# Q3. Which routes have the highest number of transactions?
df.merge(routes, on='Route_ID').groupby('Route_Name')['Transaction_ID'].count().sort_values(ascending=False)

,Transaction_ID
Route_Name,
Manchester Piccadilly_Liverpool Lime Street,4628
London Euston_Birmingham New Street,4209
London Kings Cross_York,3922
London Paddington_Reading,3873
London St Pancras_Birmingham New Street,3471
...,...
Manchester Piccadilly_Warrington,15
York_Liverpool Lime Street,15
York_Edinburgh Waverley,15


In [70]:
# Q4. Which routes generate the highest revenue?
df.merge(routes, on='Route_ID').groupby('Route_Name').agg({
    'Price':'sum',
    'Refunded_Amount':'sum',
    'Profit':'sum'
}).sort_values('Price', ascending=False)

,Price,Refunded_Amount,Profit
Route_Name,,,
London Kings Cross_York,183193,2974.00,180219.00
Liverpool Lime Street_London Euston,113299,2710.75,110588.25
London Paddington_Reading,65368,1527.00,63841.00
London Euston_Manchester Piccadilly,61004,1684.00,59320.00
London St Pancras_Birmingham New Street,52869,831.00,52038.00
...,...,...,...
Bristol Temple Meads_Cardiff Central,98,0.00,98.00
York_Leeds,78,0.00,78.00
Birmingham New Street_Wolverhampton,60,1.00,59.00


In [72]:
# Q5. Which routes have the highest delay rate?
x = trips.merge(routes, on='Route_ID')
x.groupby('Route_Name').apply(lambda g: pd.Series({
    'Total_Trips':len(g),
    'Delayed_Trips':(g['Journey_Status']=='Delayed').sum(),
    'Delay_Rate':(g['Journey_Status']=='Delayed').mean()*100
})).sort_values('Delay_Rate', ascending=False)

,Total_Trips,Delayed_Trips,Delay_Rate
Route_Name,,,
Edinburgh Waverley_London Kings Cross,43.0,43.0,100.000000
London Euston_York,16.0,16.0,100.000000
York_Wakefield,15.0,14.0,93.333333
Manchester Piccadilly_London Euston,289.0,185.0,64.013841
Liverpool Lime Street_London Euston,363.0,202.0,55.647383
...,...,...,...
York_Edinburgh,120.0,0.0,0.000000
York_Edinburgh Waverley,14.0,0.0,0.000000
York_Leeds,15.0,0.0,0.000000


In [73]:
# Q6. Which routes have the highest cancellation rate?
x.groupby('Route_Name').apply(lambda g: (g['Journey_Status']=='Cancelled').mean()*100).sort_values(ascending=False)

,0
Route_Name,
Liverpool Lime Street_Birmingham New Street,14.285714
Reading_Didcot,11.363636
London Paddington_Oxford,11.016949
Liverpool Lime Street_London St Pancras,10.344828
Birmingham New Street_London Paddington,10.000000
...,...
Reading_Liverpool Lime Street,0.000000
Reading_Birmingham New Street,0.000000
York_Edinburgh Waverley,0.000000


In [75]:
# Q7. Which routes have the highest On-Time performance rate?
x.groupby('Route_Name').apply(lambda g: (g['Journey_Status']=='On Time').mean()*100).sort_values(ascending=False)

,0
Route_Name,
Birmingham New Street_Edinburgh,100.000000
Reading_Birmingham New Street,100.000000
Birmingham New Street_London Kings Cross,100.000000
Birmingham New Street_York,100.000000
Bristol Temple Meads_Cardiff Central,100.000000
...,...
Liverpool Lime Street_London Euston,39.393939
Manchester Piccadilly_London Euston,34.948097
York_Wakefield,6.666667


In [76]:
# Q8. What is the most common delay reason per route?
trips[trips['Reason_for_Delay']!='No Delay'].merge(routes, on='Route_ID') \
.groupby(['Route_Name','Reason_for_Delay']).size() \
.reset_index(name='Count') \
.sort_values('Count', ascending=False) \
.groupby('Route_Name').head(1)

,Route_Name,Reason_for_Delay,Count
102,Manchester Piccadilly_London Euston,Weather,107
94,Manchester Piccadilly_Liverpool Lime Street,Staffing,90
38,Liverpool Lime Street_London Euston,Weather,78
16,Birmingham New Street_Manchester Piccadilly,Technical Issue,68
77,London Paddington_Reading,Staffing,66
44,Liverpool Lime Street_Manchester Piccadilly,Technical Issue,62
48,London Euston_Birmingham New Street,Signal Failure,58
64,London Kings Cross_York,Signal Failure,54
91,Manchester Piccadilly_Leeds,Signal Failure,48
29,Edinburgh Waverley_London Kings Cross,Staffing,43


In [77]:
# Q9. How many total trips are there?
print(len(trips))

19871


In [78]:
# Q10. What is the distribution of trip status?
print(trips['Journey_Status'].value_counts(normalize=True)*100)

Journey_Status
On Time      90.750340
Delayed       5.274017
Cancelled     3.975643
Name: proportion, dtype: float64


In [79]:
# Q11. How are trips distributed across weekdays vs weekends?
trips['Day_Type'] = np.where(trips['Date_of_Journey'].dt.dayofweek<5,'Weekday','Weekend')
print(trips['Day_Type'].value_counts(normalize=True)*100)

Day_Type
Weekday    71.53641
Weekend    28.46359
Name: proportion, dtype: float64


In [80]:
# Q12. How many trips occur during Peak vs Off-Peak periods?
print(trips['Peak_Category'].value_counts(normalize=True)*100)

Peak_Category
Off-Peak    62.236425
Peak        37.763575
Name: proportion, dtype: float64


In [81]:
# Q13. What is the average trip duration?
print(trips['Journey_Duration_Minutes'].mean())

70.20154999748377


In [82]:
# Q14. What are the shortest and longest trip durations?
print(trips['Journey_Duration_Minutes'].max(), trips['Journey_Duration_Minutes'].min())

270.0 15.0


In [83]:
# Q15. How does trip duration affect delay rate?
print(trips.groupby(pd.cut(trips['Journey_Duration_Minutes'], [0,60,120,180,999]))
      .apply(lambda g: (g['Journey_Status']=='Delayed').mean()*100))

Journey_Duration_Minutes
(0, 60]        3.956020
(60, 120]      4.460341
(120, 180]    22.790202
(180, 999]    28.859060
dtype: float64


In [84]:
# Q16. What is the total number of transactions?
print(len(df))

31653


In [85]:
# Q17. Transactions distribution by year
print(df.groupby(df['Date_of_Purchase'].dt.year).size())

Date_of_Purchase
2023       34
2024    31619
dtype: int64


In [86]:
# Q18. Transactions by day of month
print(df.groupby(df['Date_of_Purchase'].dt.day).size())

Date_of_Purchase
1      441
2     1336
3     1314
4     1173
5     1194
6     1219
7     1191
8     1267
9     1206
10    1155
11    1159
12    1132
13    1130
14    1146
15    1083
16    1024
17    1022
18    1116
19    1006
20     973
21     959
22     962
23     914
24     965
25     928
26     873
27     861
28     843
29     908
30     416
31     737
dtype: int64


In [88]:
# Q19. Transactions by day of week
print(df.groupby(df['Date_of_Purchase'].dt.day_name()).size())

Date_of_Purchase
Friday       4627
Monday       4455
Saturday     4477
Sunday       4676
Thursday     4362
Tuesday      4454
Wednesday    4602
dtype: int64


In [89]:
# Q20. Busiest hours of transactions
print(df.groupby(pd.to_datetime(df['Time_of_Purchase']).dt.hour).size())

Time_of_Purchase
0      925
1     1032
2      642
3     1107
4      924
5     1566
6     1910
7     2046
8     2008
9     2070
10    1187
11     743
12    1025
13     754
14    1869
15    1468
16    1056
17    2740
18    1425
19    1160
20    2239
21     573
22     458
23     726
dtype: int64


In [90]:
# Q21. Most common purchase type
print(df['Purchase_Type'].value_counts(normalize=True)*100)

Purchase_Type
Online     58.512621
Station    41.487379
Name: proportion, dtype: float64


In [91]:
# Q22. Most common payment method
print(df['Payment_Method'].value_counts(normalize=True)*100)

Payment_Method
Credit Card    60.455565
Contactless    34.227403
Debit Card      5.317032
Name: proportion, dtype: float64


In [92]:
# Q23. Most popular ticket class
print(df['Ticket_Class'].value_counts(normalize=True)*100)

Ticket_Class
Standard       90.338988
First Class     9.661012
Name: proportion, dtype: float64


In [93]:
# Q24. Most popular ticket type
print(df['Ticket_Type'].value_counts(normalize=True)*100)

Ticket_Type
Advance     55.479733
Off-Peak    27.649828
Anytime     16.870439
Name: proportion, dtype: float64


In [94]:
# Q25. Most common railcard
print(df[df['Railcard']!='None']['Railcard'].value_counts(normalize=True)*100)

Railcard
Adult       45.142059
Disabled    28.775035
Senior      26.082906
Name: proportion, dtype: float64


In [95]:
# Q26. Railcard vs non-railcard comparison
print(df.groupby(np.where(df['Railcard']=='None','No Railcard','Has Railcard')).agg({
    'Transaction_ID':'count',
    'Trip_ID':'nunique'
}))

              Transaction_ID  Trip_ID
Has Railcard           10735     6960
No Railcard            20918    13925


In [96]:
# Q27. Total revenue
print(df['Price'].sum())

741921


In [97]:
# Q28. Total profit
print(df['Profit'].sum())

726623.25


In [98]:
# Q29. Average ticket price
print(df['Price'].mean())

23.439200075822196


In [99]:
# Q30. Average profit per transaction
print(df['Profit'].mean())

22.95590465358734


In [100]:
# Q31. Revenue by ticket type
print(df.groupby('Ticket_Type')[['Price','Refunded_Amount','Profit']].sum())

              Price  Refunded_Amount     Profit
Ticket_Type                                    
Advance      309274          5527.00  303747.00
Anytime      209309          3849.75  205459.25
Off-Peak     223338          5921.00  217417.00


In [101]:
# Q32. Revenue by ticket class
print(df.groupby('Ticket_Class')[['Price','Refunded_Amount','Profit']].sum())

               Price  Refunded_Amount     Profit
Ticket_Class                                    
First Class   149399          2512.00  146887.00
Standard      592522         12785.75  579736.25


In [102]:
# Q33. Revenue by payment method
print(df.groupby('Payment_Method')[['Price','Refunded_Amount','Profit']].sum())

                 Price  Refunded_Amount     Profit
Payment_Method                                    
Contactless     219444          4207.00  215237.00
Credit Card     469511          7981.50  461529.50
Debit Card       52966          3109.25   49856.75


In [103]:
# Q34. Revenue by purchase type
print(df.groupby('Purchase_Type')[['Price','Refunded_Amount','Profit']].sum())

                Price  Refunded_Amount     Profit
Purchase_Type                                    
Online         382754          7363.50  375390.50
Station        359167          7934.25  351232.75


In [104]:
# Q35. Revenue by route
print(df.merge(routes, on='Route_ID').groupby('Route_Name')[['Price','Refunded_Amount','Profit']].sum())

                                             Price  Refunded_Amount   Profit
Route_Name                                                                  
Birmingham New Street_Coventry                 269              6.0    263.0
Birmingham New Street_Edinburgh                798              0.0    798.0
Birmingham New Street_Liverpool Lime Street   1790             16.0   1774.0
Birmingham New Street_London Euston           3594             71.5   3522.5
Birmingham New Street_London Kings Cross       535              0.0    535.0
...                                            ...              ...      ...
York_Edinburgh Waverley                        404              0.0    404.0
York_Leeds                                      78              0.0     78.0
York_Liverpool Lime Street                     323             15.0    308.0
York_Peterborough                            10764            232.0  10532.0
York_Wakefield                                 148              0.0    148.0

In [105]:
# Q36. Total refunded amount
print(df['Refunded_Amount'].sum())

15297.75


In [106]:
# Q37. Refund percentage
print((df['Refunded_Amount'].sum()/df['Price'].sum())*100)

2.0619109042606962


In [108]:
# Q38. Refund request percentage
print(df['Refund_Request'].value_counts(normalize=True)*100)

Refund_Request
No     96.467949
Yes     3.532051
Name: proportion, dtype: float64


In [109]:
# Q39. Refund rate by ticket type
print(df.groupby('Ticket_Type')['Refunded_Amount'].apply(lambda x:(x>0).mean()*100))

Ticket_Type
Advance     1.867775
Anytime     2.172285
Off-Peak    3.324954
Name: Refunded_Amount, dtype: float64


In [110]:
# Q40. Routes with highest refunds
print(df.merge(routes, on='Route_ID').groupby('Route_Name')['Refunded_Amount'].sum())

Route_Name
Birmingham New Street_Coventry                   6.0
Birmingham New Street_Edinburgh                  0.0
Birmingham New Street_Liverpool Lime Street     16.0
Birmingham New Street_London Euston             71.5
Birmingham New Street_London Kings Cross         0.0
                                               ...  
York_Edinburgh Waverley                          0.0
York_Leeds                                       0.0
York_Liverpool Lime Street                      15.0
York_Peterborough                              232.0
York_Wakefield                                   0.0
Name: Refunded_Amount, Length: 65, dtype: float64


In [113]:
# Q41. Delay impact on refund
print(df.groupby(pd.cut(df['Delay_Minutes'],[-1,0,15,30,60,999]))
      ['Refunded_Amount'].apply(lambda x:(x>0).mean()*100))

Delay_Minutes
(-1, 0]       0.000000
(0, 15]       1.157407
(15, 30]     21.980198
(30, 60]      4.635108
(60, 999]     0.000000
Name: Refunded_Amount, dtype: float64


In [114]:
# Q42. Overall delay performance
print(trips['Delay_Minutes'].agg(['sum','mean','max','min']))

sum     40066.000000
mean        2.099785
max       180.000000
min         0.000000
Name: Delay_Minutes, dtype: float64


In [115]:
# Q43. Most common delay reasons
print(trips[trips['Journey_Status']=='Delayed']['Reason_for_Delay'].value_counts(normalize=True)*100)

Reason_for_Delay
Weather            26.335878
Technical Issue    25.000000
Staffing           23.091603
Signal Failure     20.706107
Traffic             4.866412
Name: proportion, dtype: float64


In [116]:
# Q44. Most common cancellation reasons
print(trips[trips['Journey_Status']=='Cancelled']['Reason_for_Delay'].value_counts(normalize=True)*100)

Reason_for_Delay
Signal Failure     26.202532
Staffing           24.303797
Weather            23.544304
Technical Issue    13.291139
Traffic            12.658228
Name: proportion, dtype: float64


In [117]:
# Q45. Delay distribution
print(trips[trips['Journey_Status']=='Delayed'].groupby(
    pd.cut(trips['Delay_Minutes'],[0,15,30,60,999])
).size())

Delay_Minutes
(0, 15]      280
(15, 30]     304
(30, 60]     332
(60, 999]    132
dtype: int64


In [118]:
# Q46. Delay phase contribution
print(trips.groupby(pd.cut(trips['Delay_Minutes'],[0,15,30,60,999]))['Delay_Minutes'].sum())

Delay_Minutes
(0, 15]       2294.0
(15, 30]      6804.0
(30, 60]     14623.0
(60, 999]    16345.0
Name: Delay_Minutes, dtype: float64


In [121]:
# Q47. Which routes are most affected by severe delays (60+ minutes)?

x = trips.merge(routes, on='Route_ID')

result = x.groupby('Route_Name').apply(lambda g: pd.Series({
    'Total_Trips': len(g),
    'Severe_Delays': (g['Delay_Minutes'] >= 60).sum(),
    'Severe_Delay_Rate': (g['Delay_Minutes'] >= 60).mean() * 100
}))


result = result[result['Severe_Delays'] > 0]


result = result.sort_values(['Severe_Delay_Rate','Severe_Delays'], ascending=False)

print(result)

                                             Total_Trips  Severe_Delays  \
Route_Name                                                                
Manchester Piccadilly_Leeds                        119.0           40.0   
York_Doncaster                                     134.0           10.0   
Manchester Piccadilly_Nottingham                    99.0            4.0   
Manchester Piccadilly_Liverpool Lime Street       2675.0           64.0   
Liverpool Lime Street_London Euston                363.0            3.0   
London Euston_Birmingham New Street               2515.0           18.0   
Manchester Piccadilly_London Euston                289.0            1.0   

                                             Severe_Delay_Rate  
Route_Name                                                      
Manchester Piccadilly_Leeds                          33.613445  
York_Doncaster                                        7.462687  
Manchester Piccadilly_Nottingham                      4.040404  

In [120]:
# Q48. Peak vs Off-Peak delay rate
print(trips.groupby('Peak_Category')
      .apply(lambda g:(g['Journey_Status']=='Delayed').mean()*100))

Peak_Category
Off-Peak    4.140050
Peak        7.142857
dtype: float64
